# G17 — Listening vs speaking observations of the same person (paired; analysis only; test untouched)

G16 suggested that the value of observing the target depends on *what the target is doing*: faces of B while B
speaks carried ≈ 0 bits about B's next emotion, faces while B is silent carried +0.046 bits. That comparison was
between different MCIS. G17 makes it **paired**: the same MCIS, the same person (the listener L of clip III), one
context clip in which B speaks and one in which B is silent. G15/G16 cells run first; features, predictor, folds,
information gain and episode bootstrap are unchanged. B's speaking is decided by the clip-IV voice (analysis only).

* **P1 (primary).** MCIS where L is seen in clips I and II, both voices are known, and B spoke in exactly one of the
  two clips. *Silent* = L's frames in the clip where B did not speak; *speaking* = L's frames in the clip where B
  spoke; equal frame counts. Δ = mean IG(silent) − mean IG(speaking), separate logistic regressions as in G15/G16.
  Reported also by order (silent clip later / earlier), since G16 found no time effect but the order is not balanced.
* **P2 (secondary).** One logistic regression fitted on *all* context observations of L (both types pooled; all
  frames), out of fold. IG of each observation; paired difference silent − speaking within the P1 MCIS, and the
  mean IG of each type over all observations.

**Decision rules (fixed before running).**
* P1 CI above 0 → constraint (iii) stays in the main problem statement.
* P1 > 0 with the CI including 0 → (iii) becomes a reported finding with this caveat, not part of the statement.
* P1 ≤ 0 → (iii) is dropped.

In [ ]:
!pip install -q speechbrain

In [ ]:
# ======== CONFIG ========
import os


def first_existing(*paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"none of {paths}")


DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"
SPLIT_CSV = first_existing("/kaggle/input/datasets/ptrnghieu/hi-ef-split/source_folder_split_seed42.csv",
                           "/kaggle/input/hi-ef-split/source_folder_split_seed42.csv")
G8A_DIR = first_existing("/kaggle/input/datasets/ptrnghieu/g8a-features", "/kaggle/input/g8a-features")
OUT_DIR = "/kaggle/working"

N_OUTER = 5
PCA_DIM, MAXF, MAXF_POOL = 128, 24, 32
SAME_PERSON_COS, DOMINANT_MIN_FRAC = 0.45, 0.25
VOICE_SAME_COS = 0.35        # ECAPA cosine taken as "same speaker" (as in G6a / G8b)
RUN_PERSISTENCE_DIAG = True  # reads clip-IV audio for an analysis only (never a model input)
DEBUG_PER_EPISODE = None     # e.g. 6 for a quick smoke test
LR_C, N_BOOT, EP_EXCL, EP_ALPHA = 1.0, 2000, 5, 5.0

In [ ]:
import os, json, math, random, time
import numpy as np
import pandas as pd

EMO = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
POL = ['positive', 'neutral', 'negative']
E2I = {e: i for i, e in enumerate(EMO)}
P2I = {p: i for i, p in enumerate(POL)}

# Reference numbers from the locked-split report (validation, 5-seed mean)
REPORT_REF = {'B1_full': (24.65, 35.79), 'T1_future_KL': (25.34, 35.65),
              'Frozen A recognizer (E_A)': (23.21, 33.41)}


def load_tables(annot_csv, split_csv):
    """annotation.csv has no header: 0 clip_id, 1 text, 5 polarity, 6 intensity, 7 emotion, 8 uncertainty."""
    ann = pd.read_csv(annot_csv, header=None, dtype=str).set_index(0)
    sp = pd.read_csv(split_csv, dtype=str)

    def text(c):
        t = ann.at[c, 1] if c in ann.index else None
        return t if isinstance(t, str) else ''

    for k in (1, 2, 3):
        sp[f't{k}'] = sp[f'clip{k}'].map(text)
    sp['yA'] = sp['clip3_emotion'].map(E2I)
    sp['yB'] = sp['clip4_emotion'].map(E2I)
    sp['pA'] = sp['clip3'].map(lambda c: P2I.get(ann.at[c, 5], -1))
    assert sp[['yA', 'yB']].notna().all().all(), 'missing A/B emotion labels'
    return ann, sp


def eval_rows(sp, split, unlock_test=False):
    if split == 'test' and not unlock_test:
        raise RuntimeError('Test split is locked. Set UNLOCK_TEST = True only for the final, preregistered run.')
    return sp[sp['split'] == split].reset_index(drop=True)


def war_uar(pred, y, k):
    pred, y = np.asarray(pred), np.asarray(y)
    war = (pred == y).mean() * 100
    uar = np.mean([(pred[y == c] == c).mean() * 100 for c in range(k) if (y == c).any()])
    return war, uar


def source_boot_ci(pred, y, src, k, n_boot=2000, seed=0):
    """95% CI by resampling whole source folders (episodes) with replacement."""
    pred, y, src = np.asarray(pred), np.asarray(y), np.asarray(src)
    rng = np.random.default_rng(seed)
    groups = [np.where(src == s)[0] for s in np.unique(src)]
    stats = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        stats.append(war_uar(pred[idx], y[idx], k))
    lo, hi = np.percentile(np.array(stats), [2.5, 97.5], axis=0)
    return lo, hi


def report(name, pred, y, src, k=7):
    war, uar = war_uar(pred, y, k)
    lo, hi = source_boot_ci(pred, y, src, k)
    print(f'{name:<46} UAR {uar:5.2f} [{lo[1]:5.1f},{hi[1]:5.1f}]   WAR {war:5.2f} [{lo[0]:5.1f},{hi[0]:5.1f}]')
    return {'name': name, 'UAR': uar, 'WAR': war, 'UAR_lo': lo[1], 'UAR_hi': hi[1], 'WAR_lo': lo[0], 'WAR_hi': hi[0]}


def transition_tables(train_rows, alpha=1.0):
    """P(B | E_A) and P(B | E_A, P_A) estimated on TRAIN gold pairs, add-alpha smoothing."""
    T = np.full((7, 7), alpha)
    TP = np.full((7, 3, 7), alpha)
    for a, p, b in zip(train_rows['yA'], train_rows['pA'], train_rows['yB']):
        T[a, b] += 1
        if p >= 0:
            TP[a, p, b] += 1
    return T / T.sum(1, keepdims=True), TP / TP.sum(2, keepdims=True)


def rtt_forecast(pA_emo, T, pA_pol=None, TP=None):
    """Recognize-then-Transition: B distribution from A posteriors.
    Returns hard (argmax of transition row of argmax A) and soft (expected) B predictions."""
    hard = T[pA_emo.argmax(1)].argmax(1)
    if pA_pol is not None and TP is not None:
        pB = np.einsum('na,np,apb->nb', pA_emo, pA_pol, TP)  # assumes E_A and P_A posteriors independent
    else:
        pB = pA_emo @ T
    return hard, pB.argmax(1), pB

import glob, pickle
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ANNOT_CSV = glob.glob(os.path.join(DATASET_DIR, "*", "Hi-EF", "annotation.csv"))[0]
ann, sp = load_tables(ANNOT_CSV, SPLIT_CSV)
sp = sp[sp.split.isin(['train', 'val'])].reset_index(drop=True)      # test rows are dropped here
assert 'test' not in set(sp.split)
if DEBUG_PER_EPISODE:
    sp = sp.groupby('source_folder', group_keys=False).head(DEBUG_PER_EPISODE).reset_index(drop=True)
DEV = sp
train_all = ev = DEV          # names used by the shared feature-loading cell
N = len(DEV)
EPS = np.array(sorted(DEV.source_folder.unique()))
print(f"development MCIS {N} | episodes {len(EPS)}")


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

## From G8a features to role-tagged slots

In [ ]:
from collections import defaultdict
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA

G8 = {}
for f in sorted(glob.glob(os.path.join(G8A_DIR, '**', 'shard_*.pkl'), recursive=True)):
    G8.update(pickle.load(open(f, 'rb')))
need = sorted(set(DEV[['clip1', 'clip2', 'clip3']].values.ravel()))
miss = [c for c in need if c not in G8]
assert not miss, f"{len(miss)} clips missing from G8a, e.g. {miss[:3]}"


def softmax(z):
    e = np.exp(z - z.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)


# per-clip frame features except the PCA part (18 dims), and the raw HSEmotion embeddings
FB, EMB = {}, {}
for c in need:
    fs = G8[c]['faces']
    if not fs:
        FB[c], EMB[c] = np.zeros((0, 18), np.float32), None
        continue
    dur = max(G8[c]['meta'].get('duration') or 0.0, 1e-3)
    fer = np.stack([d['fer'] for d in fs]).astype(np.float32)
    box = np.stack([d['box'] for d in fs])
    FB[c] = np.concatenate([
        softmax(fer[:, :8]), fer[:, 8:10], np.stack([d['pose'] for d in fs]) / 90.0,
        np.stack([(box[:, 0] + box[:, 2]) / 2, (box[:, 1] + box[:, 3]) / 2,
                  np.sqrt(np.clip((box[:, 2] - box[:, 0]) * (box[:, 3] - box[:, 1]), 0, None))], 1),
        np.nan_to_num(np.array([[d['mouth'] * 10] for d in fs], np.float32)),
        np.array([[min(d['t'] / dur, 1.0)] for d in fs], np.float32)], 1).astype(np.float32)
    EMB[c] = np.stack([d['fer_emb'] for d in fs]) if all('fer_emb' in d for d in fs) else None
HAS_EMB = all(EMB[c] is not None for c in need if len(G8[c]['faces']))
FDIM = (PCA_DIM if HAS_EMB else 0) + 18
print(f"HSEmotion embedding available: {HAS_EMB} | frame feature dim {FDIM}")


def pick_frames(idx, cap):
    return idx if len(idx) <= cap else [idx[i] for i in np.linspace(0, len(idx) - 1, cap).astype(int)]


def sync(c, face_ids):
    # correlation of mouth opening with the audio energy envelope, and mouth variability, for a set of faces
    A = G8[c]['audio']
    fs = [G8[c]['faces'][j] for j in face_ids]
    fs = [d for d in fs if np.isfinite(d['mouth'])]
    if A is None or len(fs) < 4:
        return 0.0, 0.0
    m = np.array([d['mouth'] for d in fs])
    env = A['env']
    e = np.array([env[min(int(d['t'] * 10), len(env) - 1)] for d in fs]) if len(env) else np.zeros(len(fs))
    r = float(np.corrcoef(m, e)[0, 1]) if m.std() > 1e-6 and e.std() > 1e-9 else 0.0
    return r, float(m.std() * 10)


def mean12(c_faces, n_sampled):
    if not c_faces:
        return np.zeros(12, np.float32)
    fer = np.stack([d['fer'] for d in c_faces]).astype(np.float32)
    v = np.concatenate([softmax(fer[:, :8]), fer[:, 8:10]], 1).mean(0)
    return np.concatenate([v, [1.0, len({d['frame'] for d in c_faces}) / max(n_sampled, 1)]]).astype(np.float32)


NVOICE = 3 * 2 + 3
SLOT, PSLOT = {}, {}                      # (n, role, clip) / (n, clip) -> (clip id, face indices)
FMASK = np.zeros((N, 3, 3, MAXF), bool); PMASK = np.zeros((N, 1, 3, MAXF_POOL), bool)
VOI = np.zeros((N, 3, NVOICE), np.float32)
LRF = np.zeros((N, 60), np.float32)       # G6b-style role means (late fusion and the joint-branch arm)
CENT_COS = np.full(N, np.nan, np.float32)
for n, row in enumerate(tqdm(DEV.itertuples(), total=N, desc='roles')):
    cl = [row.clip1, row.clip2, row.clip3]
    items = [(k, j) for k, c in enumerate(cl) for j in range(len(G8[c]['faces']))]
    lab = np.zeros(len(items), int)
    E = np.stack([G8[cl[k]]['faces'][j]['arc'] for k, j in items]).astype(np.float32) if items else None
    if len(items) > 1:
        lab = AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average',
                                      distance_threshold=1 - SAME_PERSON_COS).fit_predict(E)
    frames = defaultdict(set)
    for (k, j), p in zip(items, lab):
        frames[(k, p)].add(G8[cl[k]]['faces'][j]['frame'])
    ids3 = sorted({p for (k, p) in frames if k == 2}, key=lambda p: -len(frames[(2, p)]))
    A = ids3[0] if ids3 else None
    L = ids3[1] if len(ids3) > 1 else None
    if A is not None and L is not None:
        ca, cb = E[lab == A].mean(0), E[lab == L].mean(0)
        CENT_COS[n] = float(ca @ cb / (np.linalg.norm(ca) * np.linalg.norm(cb) + 1e-9))
    role = lambda p: 0 if p == A else (1 if p == L else 2)
    by = defaultdict(list)
    for (k, j), p in zip(items, lab):
        by[(role(p), k)].append(j)
        by[('pool', k)].append(j)
    for k, c in enumerate(cl):
        for r in range(3):
            idx = pick_frames(sorted(by[(r, k)], key=lambda j: G8[c]['faces'][j]['t']), MAXF)
            SLOT[(n, r, k)] = (c, idx); FMASK[n, r, k, :len(idx)] = True
        idx = pick_frames(sorted(by[('pool', k)], key=lambda j: G8[c]['faces'][j]['t']), MAXF_POOL)
        PSLOT[(n, k)] = (c, idx); PMASK[n, 0, k, :len(idx)] = True
        v = [x for r in range(3) for x in sync(c, by[(r, k)])]
        a3, ak = G8[cl[2]]['audio'], G8[c]['audio']
        ok = a3 is not None and ak is not None and a3.get('ecapa') is not None and ak.get('ecapa') is not None
        vcos = float(a3['ecapa'].astype(np.float32) @ ak['ecapa'].astype(np.float32)) if ok else 0.0
        VOI[n, k] = v + [vcos, float(ok), len(set(lab)) / 5.0]

    def faces_of(k, p):
        return [G8[cl[k]]['faces'][j] for (kk, j), q in zip(items, lab) if kk == k and q == p]

    def dominant(k):
        ns = G8[cl[k]]['meta']['n_sampled']
        cand = sorted({q for (kk, q) in frames if kk == k}, key=lambda q: -len(frames[(k, q)]))
        return cand[0] if cand and len(frames[(k, cand[0])]) / max(ns, 1) >= DOMINANT_MIN_FRAC else None

    n3 = G8[cl[2]]['meta']['n_sampled']
    blocks = [mean12(faces_of(2, A) if A is not None else [], n3), mean12(faces_of(2, L) if L is not None else [], n3),
              mean12((faces_of(0, L) + faces_of(1, L)) if L is not None else [], n3)]
    for k in (1, 0):
        d = dominant(k)
        blocks.append(mean12(faces_of(k, d) if d is not None else [], G8[cl[k]]['meta']['n_sampled']))
    LRF[n] = np.concatenate(blocks)

VIS = FMASK[:, 1, 2].any(-1)
print(f"A found in III {FMASK[:, 0, 2].any(-1).mean() * 100:.1f}% | listener visible in III {VIS.mean() * 100:.1f}% | "
      f"listener also in I/II {FMASK[:, 1, :2].any((-1, -2)).mean() * 100:.1f}%")


def build_face_tensors(fit_clips):
    # PCA of the HSEmotion embedding fitted on faces of the training clips of the current fold only
    pca, var = None, None
    if HAS_EMB:
        pool = np.concatenate([EMB[c] for c in fit_clips if EMB.get(c) is not None])
        pick = np.random.default_rng(0).choice(len(pool), min(60000, len(pool)), replace=False)
        pca = PCA(PCA_DIM, random_state=0).fit(pool[pick].astype(np.float32))
        var = float(pca.explained_variance_ratio_.sum())
    FV = {}
    for c in need:
        if len(FB[c]) == 0:
            FV[c] = np.zeros((0, FDIM), np.float32)
        else:
            FV[c] = np.concatenate([pca.transform(EMB[c].astype(np.float32)), FB[c]], 1) if HAS_EMB else FB[c]
    Fa = np.zeros((N, 3, 3, MAXF, FDIM), np.float16)
    Pa = np.zeros((N, 1, 3, MAXF_POOL, FDIM), np.float16)
    for (n, r, k), (c, idx) in SLOT.items():
        if idx:
            Fa[n, r, k, :len(idx)] = FV[c][idx]
    for (n, k), (c, idx) in PSLOT.items():
        if idx:
            Pa[n, 0, k, :len(idx)] = FV[c][idx]
    return torch.tensor(Fa, device=DEVICE), torch.tensor(Pa, device=DEVICE), var

## Clip IV voice (analysis only) and the G8b persistence diagnostic, re-run

In [ ]:
# (3) is emotional inertia person-specific? Clip IV voice (analysis only) says whether B spoke in clip I / II.
if RUN_PERSISTENCE_DIAG:
    import librosa
    try:
        from speechbrain.inference.speaker import EncoderClassifier
    except ImportError:
        from speechbrain.pretrained import EncoderClassifier
    spk = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir=f"{OUT_DIR}/ecapa",
                                         run_opts={"device": DEVICE})
    AUDIO_ROOTS = [os.path.join(r, 'audio') for r in glob.glob(os.path.join(DATASET_DIR, '*', 'Hi-EF'))]

    def audio_path(c):
        ep, num = c.split('/')
        for root in AUDIO_ROOTS:
            for ext in ('.mp3', '.wav', '.flac', '.m4a'):
                p = os.path.join(root, ep, num + ext)
                if os.path.exists(p):
                    return p
        return None

    V4 = {}
    for c in tqdm(sorted(set(DEV.clip4)), desc='clip IV voice (analysis only)'):
        p = audio_path(c)
        if p is None:
            continue
        try:
            wav, _ = librosa.load(p, sr=16000, mono=True)
        except Exception:
            continue
        if len(wav) < 8000:
            continue
        with torch.no_grad():
            e = spk.encode_batch(torch.tensor(wav, dtype=torch.float32).unsqueeze(0)).reshape(-1).cpu().numpy()
        V4[c] = e / (np.linalg.norm(e) + 1e-9)
    del spk; torch.cuda.empty_cache()

    def gold(c):
        e = ann.at[c, 7] if c in ann.index else None
        return E2I.get(e, -1) if isinstance(e, str) else -1

    rows_ = []
    for n, row in enumerate(DEV.itertuples()):
        if row.clip4 not in V4:
            continue
        for k, (name, c) in enumerate((('I', row.clip1), ('II', row.clip2))):
            a, y = G8[c]['audio'], gold(c)
            if a is None or a.get('ecapa') is None or y < 0:
                continue
            isB = float(a['ecapa'].astype(np.float32) @ V4[row.clip4].astype(np.float32)) >= VOICE_SAME_COS
            rows_.append({'ep': row.source_folder, 'ctx': name, 'B_spoke': bool(isB), 'same': bool(y == row.yB),
                          'A_spoke': bool(VOI[n, k, 6] >= VOICE_SAME_COS)})
    pdf = pd.DataFrame(rows_, columns=['ep', 'ctx', 'B_spoke', 'same', 'A_spoke'])
    print(f"(3) clip IV voice found for {len(V4)}/{DEV.clip4.nunique()} clips; labelled context clips analysed: {len(pdf)}")
    rng_ = np.random.default_rng(0)
    for name in ('I', 'II'):
        d = pdf[pdf['ctx'] == name]
        if d.B_spoke.sum() < 10 or (~d.B_spoke).sum() < 10:
            print(f"  clip {name}: too few rows"); continue
        eps_ = d.ep.unique()
        diffs = []
        for _ in range(2000):
            s = pd.concat([d[d.ep == e] for e in rng_.choice(eps_, len(eps_))])
            if s.B_spoke.any() and (~s.B_spoke).any():
                diffs.append(s[s.B_spoke].same.mean() - s[~s.B_spoke].same.mean())
        lo, hi = np.percentile(diffs, [2.5, 97.5]) * 100
        third = d[~d.B_spoke & ~d.A_spoke]
        print(f"  clip {name}: P(y_IV = y_{name}) when B spoke {d[d.B_spoke].same.mean() * 100:.1f}% (n={d.B_spoke.sum()}) | "
              f"when someone else spoke {d[~d.B_spoke].same.mean() * 100:.1f}% (n={(~d.B_spoke).sum()}; third person only "
              f"{third.same.mean() * 100 if len(third) else float('nan'):.1f}%, n={len(third)}) | difference CI [{lo:+.1f}, {hi:+.1f}]")
    print("  -> a clearly positive difference supports a separate B-inertia path; ~0 supports one merged persistence path")

## Gates

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize_scalar

# folds exactly as G8b-G14
sizes = DEV.source_folder.value_counts()
order = list(sizes.index)
random.Random(0).shuffle(order)
order = sorted(order, key=lambda e: -sizes[e])
load_, FOLD = [0] * N_OUTER, {}
for e in order:
    f = int(np.argmin(load_)); FOLD[e] = f; load_[f] += sizes[e]
fold = DEV.source_folder.map(FOLD).values
src = DEV.source_folder.values
y = DEV.yB.values.astype(int)
DUR = np.array([[max(G8[c]['meta'].get('duration') or 0.0, 1e-3) for c in r]
                for r in DEV[['clip1', 'clip2', 'clip3']].values])
LN2 = np.log(2)


def expr(c, idx):
    # one observation: mean over frames of the 8 HSEmotion probabilities, valence, arousal
    fer = np.stack([G8[c]['faces'][j]['fer'] for j in idx]).astype(np.float32)
    return np.concatenate([softmax(fer[:, :8]), fer[:, 8:10]], 1).mean(0)


def dist_to_iv(n, k, c, idx):
    # mean time (s) from the chosen frames to the end of clip III, i.e. to the start of clip IV
    t = np.array([G8[c]['faces'][j]['t'] for j in idx], np.float32)
    return float(np.mean(DUR[n, k] - t) + DUR[n, k + 1:3].sum())


def oof_ig(X, rows):
    # out-of-fold information gain (nats) of a logistic regression, and logit-adjusted predictions
    ig, pred = np.full(len(rows), np.nan), np.full(len(rows), -1)
    yr, fr = y[rows], fold[rows]
    for f in range(N_OUTER):
        tr, te = fr != f, fr == f
        if te.sum() == 0 or len(np.unique(yr[tr])) < 2:
            continue
        pri = np.bincount(yr[tr], minlength=7) + 1.0; pri /= pri.sum()
        sc = StandardScaler().fit(X[tr])
        m = LogisticRegression(C=LR_C, max_iter=3000).fit(sc.transform(X[tr]), yr[tr])
        P = np.full((te.sum(), 7), 1e-6); P[:, m.classes_] = m.predict_proba(sc.transform(X[te]))
        P /= P.sum(1, keepdims=True)
        ig[te] = np.log(P[np.arange(te.sum()), yr[te]]) - np.log(pri[yr[te]])
        pred[te] = (np.log(P) - np.log(pri)).argmax(1)
    return ig, pred


def ep_boot(stat, s, n_boot=N_BOOT, seed=0):
    # stat(idx) -> number; 95% interval over episodes resampled with replacement
    rng = np.random.default_rng(seed)
    groups = [np.where(s == e)[0] for e in np.unique(s)]
    v = [stat(np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])) for _ in range(n_boot)]
    v = np.array(v, float); v = v[np.isfinite(v)]
    return np.percentile(v, [2.5, 97.5]) if len(v) else np.array([np.nan, np.nan])


def verdict(d, lo):
    return 'PASS' if lo > 0 else ('DIRECTIONAL' if d > 0 else 'FAIL')


PER = pd.DataFrame({'sample_id': DEV.sample_id.values, 'src': src, 'fold': fold, 'y': y,
                    'dur1': DUR[:, 0], 'dur2': DUR[:, 1], 'dur3': DUR[:, 2]})
for k in range(3):
    PER[f'nL{k + 1}'] = [len(SLOT[(n, 1, k)][1]) for n in range(N)]
PER['L_last_gap3'] = [DUR[n, 2] - max(G8[SLOT[(n, 1, 2)][0]]['faces'][j]['t'] for j in SLOT[(n, 1, 2)][1])
                      if SLOT[(n, 1, 2)][1] else np.nan for n in range(N)]
RES = {}

# ---------------- Gate 1a: near (clip III) vs far (latest of II / I), same MCIS ----------------
rows, Xn, Xf, Xn_all, Xf_all, dn, dfar = [], [], [], [], [], [], []
for n in range(N):
    cn, idn = SLOT[(n, 1, 2)]
    kf = next((k for k in (1, 0) if SLOT[(n, 1, k)][1]), None)
    if not idn or kf is None:
        continue
    cf, idf = SLOT[(n, 1, kf)]
    m = min(len(idn), len(idf))
    pn, pf = pick_frames(idn, m), pick_frames(idf, m)
    rows.append(n); Xn.append(expr(cn, pn)); Xf.append(expr(cf, pf)); Xn_all.append(expr(cn, idn)); Xf_all.append(expr(cf, idf))
    dn.append(dist_to_iv(n, 2, cn, pn)); dfar.append(dist_to_iv(n, kf, cf, pf))
rows = np.array(rows, int)
print(f"G1a: {len(rows)} MCIS with L in clip III and in clip I/II | median time to clip IV: near {np.median(dn):.2f}s, "
      f"far {np.median(dfar):.2f}s" if len(rows) else "G1a: no MCIS")
if len(rows) >= 50:
    ign, pn_ = oof_ig(np.array(Xn), rows); igf, pf_ = oof_ig(np.array(Xf), rows)
    ign_a, _ = oof_ig(np.array(Xn_all), rows); igf_a, _ = oof_ig(np.array(Xf_all), rows)
    ok = np.isfinite(ign) & np.isfinite(igf)
    s1, d1 = src[rows][ok], (ign - igf)[ok]
    D1 = d1.mean() / LN2
    lo1, hi1 = ep_boot(lambda i: d1[i].mean(), s1) / LN2
    lon, hin = ep_boot(lambda i: ign[ok][i].mean(), s1) / LN2
    lof, hif = ep_boot(lambda i: igf[ok][i].mean(), s1) / LN2
    da = (ign_a - igf_a)[ok]
    loa, hia = ep_boot(lambda i: da[i].mean(), s1) / LN2
    un, uf = war_uar(pn_[ok], y[rows][ok], 7)[1], war_uar(pf_[ok], y[rows][ok], 7)[1]
    print(f"  IG near {ign[ok].mean() / LN2:+.4f} bits [{lon:+.4f},{hin:+.4f}] | IG far {igf[ok].mean() / LN2:+.4f} bits "
          f"[{lof:+.4f},{hif:+.4f}]")
    print(f"  Δ1 = near − far {D1:+.4f} bits [{lo1:+.4f},{hi1:+.4f}] (equal frame counts, primary)")
    print(f"  all frames: Δ {da.mean() / LN2:+.4f} bits [{loa:+.4f},{hia:+.4f}] | UAR (logit-adjusted) near {un:.2f}, far {uf:.2f}")
    g1 = 'UNINFORMATIVE' if lon <= 0 else verdict(D1, lo1)
    RES['G1a'] = dict(n=int(ok.sum()), IG_near=ign[ok].mean() / LN2, IG_far=igf[ok].mean() / LN2, delta=D1, lo=lo1, hi=hi1,
                      delta_allframes=da.mean() / LN2, med_s_near=float(np.median(dn)), med_s_far=float(np.median(dfar)),
                      verdict=g1)
    PER.loc[rows, 'IG1_near'] = ign / LN2; PER.loc[rows, 'IG1_far'] = igf / LN2
    PER.loc[rows, 's_near'] = dn; PER.loc[rows, 's_far'] = dfar
else:
    g1 = 'UNINFORMATIVE'
    RES['G1a'] = dict(n=int(len(rows)), verdict=g1)
print(f"  GATE 1 (G1a): {g1}")

# ---------------- Gate 1b: late vs early half of L's frames in clip III ----------------
rows_b, Xe, Xl = [], [], []
for n in range(N):
    c, idx = SLOT[(n, 1, 2)]
    if len(idx) >= 4:
        h = len(idx) // 2
        rows_b.append(n); Xe.append(expr(c, idx[:h])); Xl.append(expr(c, idx[-h:]))
rows_b = np.array(rows_b, int)
if len(rows_b) >= 50:
    ige, _ = oof_ig(np.array(Xe), rows_b); igl, _ = oof_ig(np.array(Xl), rows_b)
    ok = np.isfinite(ige) & np.isfinite(igl)
    db = (igl - ige)[ok]
    lob, hib = ep_boot(lambda i: db[i].mean(), src[rows_b][ok]) / LN2
    print(f"G1b: {ok.sum()} MCIS | IG late {igl[ok].mean() / LN2:+.4f}, early {ige[ok].mean() / LN2:+.4f} bits | "
          f"late − early {db.mean() / LN2:+.4f} [{lob:+.4f},{hib:+.4f}]")
    RES['G1b'] = dict(n=int(ok.sum()), delta=db.mean() / LN2, lo=lob, hi=hib)
else:
    print(f"G1b: too few MCIS ({len(rows_b)})")

# ---------------- Gate 2: persistence of B's last labelled emotion ----------------
BSP = np.zeros((N, 2), bool); YCTX = np.full((N, 2), -1)
for n, row in enumerate(DEV.itertuples()):
    v4 = V4.get(row.clip4)
    for k, c in enumerate((row.clip1, row.clip2)):
        YCTX[n, k] = gold(c)
        a = G8[c]['audio']
        if v4 is not None and a is not None and a.get('ecapa') is not None:
            BSP[n, k] = float(a['ecapa'].astype(np.float32) @ v4.astype(np.float32)) >= VOICE_SAME_COS
PER['B_spoke_I'], PER['B_spoke_II'], PER['y_I'], PER['y_II'] = BSP[:, 0], BSP[:, 1], YCTX[:, 0], YCTX[:, 1]

# episode baseline from the episode's other labelled clips (|clip number - clip IV number| > EP_EXCL)
lab = ann[7].map(E2I)
lab = lab[lab.notna()].astype(int)
lab_ep = pd.Series([i.split('/')[0] for i in lab.index], index=lab.index)
lab_num = pd.Series([int(i.split('/')[1]) for i in lab.index], index=lab.index)
dev_eps = set(DEV.source_folder)
glob_p = np.bincount(lab[lab_ep.isin(dev_eps)].values, minlength=7) + 1.0; glob_p /= glob_p.sum()
PI_EP = np.zeros((N, 7))
for n, row in enumerate(DEV.itertuples()):
    ep, num4 = row.clip4.split('/')[0], int(row.clip4.split('/')[1])
    sel = (lab_ep.values == ep) & (np.abs(lab_num.values - num4) > EP_EXCL)
    cnt = np.bincount(lab.values[sel], minlength=7)
    PI_EP[n] = (cnt + EP_ALPHA * glob_p) / (cnt.sum() + EP_ALPHA)

near = BSP[:, 1] & (YCTX[:, 1] >= 0)
far = BSP[:, 0] & ~BSP[:, 1] & (YCTX[:, 0] >= 0)
YLAST = np.where(near, YCTX[:, 1], np.where(far, YCTX[:, 0], -1))
print(f"\nG2: B spoke in clip II (near) {near.sum()} MCIS | in clip I only (far) {far.sum()} MCIS | "
      f"median gap near {np.median(DUR[near, 2]) if near.any() else float('nan'):.2f}s, "
      f"far {np.median(DUR[far, 1:].sum(1)) if far.any() else float('nan'):.2f}s")


def fit_scalar(nll):
    return minimize_scalar(nll, bounds=(-5, 5), method='bounded').x


def persist_logp(beta, yl, pri):
    lp = np.log(pri)[None].repeat(len(yl), 0); lp[np.arange(len(yl)), yl] += beta
    return lp - np.log(np.exp(lp).sum(1, keepdims=True))


def base_logp(gam, pe, pri):
    lp = np.log(pri)[None] + gam * (np.log(pe) - np.log(pri)[None])
    return lp - np.log(np.exp(lp).sum(1, keepdims=True))


def fit_beta(r, yl):
    pri = np.bincount(y[r], minlength=7) + 1.0; pri /= pri.sum()
    return fit_scalar(lambda b: -persist_logp(b, yl, pri)[np.arange(len(r)), y[r]].sum())


def oof_group(r):
    # out-of-fold IG (nats) of the persistence model and of the episode-baseline model
    igl, ige = np.full(len(r), np.nan), np.full(len(r), np.nan)
    for f in range(N_OUTER):
        tr, te = fold[r] != f, fold[r] == f
        if te.sum() == 0 or tr.sum() < 10:
            continue
        a, b = r[tr], r[te]
        pri = np.bincount(y[a], minlength=7) + 1.0; pri /= pri.sum()
        bet = fit_scalar(lambda v: -persist_logp(v, YLAST[a], pri)[np.arange(len(a)), y[a]].sum())
        gam = fit_scalar(lambda v: -base_logp(v, PI_EP[a], pri)[np.arange(len(a)), y[a]].sum())
        igl[te] = persist_logp(bet, YLAST[b], pri)[np.arange(len(b)), y[b]] - np.log(pri[y[b]])
        ige[te] = base_logp(gam, PI_EP[b], pri)[np.arange(len(b)), y[b]] - np.log(pri[y[b]])
    return igl, ige


rn, rf = np.where(near)[0], np.where(far)[0]
if len(rn) >= 30 and len(rf) >= 30:
    bn, bf = fit_beta(rn, YLAST[rn]), fit_beta(rf, YLAST[rf])
    allr = np.concatenate([rn, rf]); isn = np.r_[np.ones(len(rn), bool), np.zeros(len(rf), bool)]

    def dbeta(i):
        a, b = allr[i][isn[i]], allr[i][~isn[i]]
        if len(a) < 10 or len(b) < 10:
            return np.nan
        return fit_beta(a, YLAST[a]) - fit_beta(b, YLAST[b])

    lo2, hi2 = ep_boot(dbeta, src[allr], n_boot=1000)
    g2 = verdict(bn - bf, lo2)
    print(f"  G2a persistence β: near {bn:+.3f} | far {bf:+.3f} | Δβ {bn - bf:+.3f} [{lo2:+.3f},{hi2:+.3f}]  -> GATE 2: {g2}")
    print(f"      P(y_IV = y_last): near {np.mean(y[rn] == YLAST[rn]) * 100:.1f}% | far {np.mean(y[rf] == YLAST[rf]) * 100:.1f}%")
    igl_n, ige_n = oof_group(rn); igl_f, ige_f = oof_group(rf)
    PER.loc[rn, 'IG2_last'] = igl_n / LN2; PER.loc[rn, 'IG2_ep'] = ige_n / LN2
    PER.loc[rf, 'IG2_last'] = igl_f / LN2; PER.loc[rf, 'IG2_ep'] = ige_f / LN2
    PER['G2_group'] = np.where(near, 'near', np.where(far, 'far', ''))
    vv = np.r_[ige_n - igl_n, ige_f - igl_f]
    ok = np.isfinite(vv)

    def dd(i):
        i = i[ok[i]]
        a, b = vv[i][~isn[i]], vv[i][isn[i]]
        return a.mean() - b.mean() if len(a) and len(b) else np.nan

    Dv = dd(np.arange(len(allr)))
    lod, hid = ep_boot(dd, src[allr])
    print(f"  G2b OOF IG (bits): near last {np.nanmean(igl_n) / LN2:+.4f}, episode {np.nanmean(ige_n) / LN2:+.4f} | "
          f"far last {np.nanmean(igl_f) / LN2:+.4f}, episode {np.nanmean(ige_f) / LN2:+.4f}")
    hb = 'context-dependent' if lod > 0 else 'global'
    print(f"      D = [ep − last]_far − [ep − last]_near {Dv / LN2:+.4f} bits [{lod / LN2:+.4f},{hid / LN2:+.4f}] "
          f"-> home base: {hb}")
    RES['G2a'] = dict(n_near=len(rn), n_far=len(rf), beta_near=bn, beta_far=bf, delta=bn - bf, lo=lo2, hi=hi2, verdict=g2)
    RES['G2b'] = dict(D_bits=Dv / LN2, lo=lod / LN2, hi=hid / LN2, home_base=hb)
else:
    g2 = 'UNINFORMATIVE'
    print(f"  too few MCIS for gate 2 (near {len(rn)}, far {len(rf)})")
    RES['G2a'] = dict(n_near=len(rn), n_far=len(rf), verdict=g2)

both = BSP.all(1) & (YCTX >= 0).all(1)
if both.sum() >= 30:
    r = np.where(both)[0]
    b2, b1 = fit_beta(r, YCTX[r, 1]), fit_beta(r, YCTX[r, 0])
    lob2, hib2 = ep_boot(lambda i: fit_beta(r[i], YCTX[r[i], 1]) - fit_beta(r[i], YCTX[r[i], 0]), src[r], n_boot=1000)
    print(f"  G2a within MCIS (B spoke in I and II, n={both.sum()}): β(y_II) {b2:+.3f} vs β(y_I) {b1:+.3f}, "
          f"Δ {b2 - b1:+.3f} [{lob2:+.3f},{hib2:+.3f}]")
    RES['G2a_within'] = dict(n=int(both.sum()), beta_II=b2, beta_I=b1, delta=b2 - b1, lo=lob2, hi=hib2)

decision = ('BUILD the latent-dynamics model' if g1 == 'PASS' and g2 == 'PASS' else
            'STOP this formulation' if 'FAIL' in (g1, g2) else 'REPORT, authors decide')
print(f"\n== GATE 1: {g1} | GATE 2: {g2} | decision (fixed rule): {decision} ==")
PER.to_csv(f"{OUT_DIR}/g15_per_mcis.csv", index=False)
json.dump({k: {a: (float(b) if isinstance(b, (np.floating, float)) else b) for a, b in v.items()} for k, v in RES.items()}
          | {'decision': decision}, open(f"{OUT_DIR}/g15_gates.json", 'w'), indent=1, default=str)
print(f"saved {OUT_DIR}/g15_per_mcis.csv and g15_gates.json")

## G16 contrasts

In [ ]:
VOK = np.zeros((N, 2), bool)
for n, row in enumerate(DEV.itertuples()):
    for k, c in enumerate((row.clip1, row.clip2)):
        a = G8[c]['audio']
        VOK[n, k] = row.clip4 in V4 and a is not None and a.get('ecapa') is not None
SILENT = VOK & ~BSP                                     # voice known, and B did not speak in clip k
RES16 = {}


def contrast(name, rows, Xa, Xb, la, lb):
    rows = np.array(rows, int)
    if len(rows) < 50:
        print(f"{name}: too few MCIS ({len(rows)})"); RES16[name] = dict(n=int(len(rows))); return None
    ia, _ = oof_ig(np.array(Xa), rows); ib, _ = oof_ig(np.array(Xb), rows)
    ok = np.isfinite(ia) & np.isfinite(ib)
    d, s_ = (ia - ib)[ok], src[rows][ok]
    lo, hi = ep_boot(lambda i: d[i].mean(), s_) / LN2
    print(f"{name}: n={ok.sum()} | IG {la} {ia[ok].mean() / LN2:+.4f}, {lb} {ib[ok].mean() / LN2:+.4f} bits | "
          f"{la} − {lb} {d.mean() / LN2:+.4f} [{lo:+.4f},{hi:+.4f}]")
    RES16[name] = dict(n=int(ok.sum()), IG_a=ia[ok].mean() / LN2, IG_b=ib[ok].mean() / LN2, delta=d.mean() / LN2,
                       lo=lo, hi=hi)
    return d.mean() / LN2, lo


def pair(n, ka, kb):
    (ca, ia_), (cb, ib_) = SLOT[(n, 1, ka)], SLOT[(n, 1, kb)]
    m = min(len(ia_), len(ib_))
    return expr(ca, pick_frames(ia_, m)), expr(cb, pick_frames(ib_, m))


# T1 / T1-all: clip II vs clip I
r1, a1, b1, r1a, a1a, b1a = [], [], [], [], [], []
for n in range(N):
    if SLOT[(n, 1, 0)][1] and SLOT[(n, 1, 1)][1]:
        xa, xb = pair(n, 1, 0)
        r1a.append(n); a1a.append(xa); b1a.append(xb)
        if SILENT[n].all():
            r1.append(n); a1.append(xa); b1.append(xb)
t1 = contrast('T1 (B silent in I and II)', r1, a1, b1, 'II', 'I')
contrast('T1-all', r1a, a1a, b1a, 'II', 'I')

# T2: clip III vs the latest far clip, where B did not speak in that far clip
r2, a2, b2 = [], [], []
for n in range(N):
    kf = next((k for k in (1, 0) if SLOT[(n, 1, k)][1]), None)
    if SLOT[(n, 1, 2)][1] and kf is not None and SILENT[n, kf]:
        xa, xb = pair(n, 2, kf)
        r2.append(n); a2.append(xa); b2.append(xb)
t2 = contrast('T2 (far clip non-speaking)', r2, a2, b2, 'III', 'far')

# T3: G15's far observations, split by whether B spoke in that far clip
if 'IG1_far' in PER:
    far_k = np.array([next((k for k in (1, 0) if SLOT[(n, 1, k)][1]), -1) for n in range(N)])
    has = PER['IG1_far'].notna().values & (far_k >= 0)
    kk = np.where(has, far_k, 0)
    known = has & VOK[np.arange(N), kk]
    spoke = known & BSP[np.arange(N), kk]
    quiet = known & ~BSP[np.arange(N), kk]
    for nm, m in (('far clip, B spoke', spoke), ('far clip, B silent', quiet)):
        print(f"T3 {nm}: n={m.sum()} | IG far {PER.loc[m, 'IG1_far'].mean():+.4f} bits | "
              f"IG near {PER.loc[m, 'IG1_near'].mean():+.4f} bits")
        RES16[f'T3 {nm}'] = dict(n=int(m.sum()), IG_far=float(PER.loc[m, 'IG1_far'].mean()),
                                 IG_near=float(PER.loc[m, 'IG1_near'].mean()))

if t1 is None:
    dec = 'T1 UNINFORMATIVE (too few MCIS)'
elif t1[1] > 0:
    dec = 'time matters with the type held -> continuous-time latent state justified'
elif t1[0] > 0:
    dec = 'DIRECTIONAL -> authors decide'
else:
    dec = 'no decay beyond the clip-III reaction -> discrete-step state, not OU'
print(f"\n== G16 decision (fixed rule): {dec} ==")
PER['B_voice_known_I'], PER['B_voice_known_II'] = VOK[:, 0], VOK[:, 1]
PER.to_csv(f"{OUT_DIR}/g16_per_mcis.csv", index=False)
json.dump({k: {a: (float(b) if isinstance(b, (np.floating, float)) else b) for a, b in v.items()} for k, v in RES16.items()}
          | {'decision': dec}, open(f"{OUT_DIR}/g16_results.json", 'w'), indent=1, default=str)
print(f"saved {OUT_DIR}/g16_per_mcis.csv and g16_results.json")

## G17 contrasts

In [ ]:
RES17 = {}
# P1: same MCIS, one silent and one speaking context clip
rP, xs, xk, order = [], [], [], []
for n in range(N):
    if SLOT[(n, 1, 0)][1] and SLOT[(n, 1, 1)][1] and VOK[n].all() and BSP[n].sum() == 1:
        ks, kk = int(np.where(~BSP[n])[0][0]), int(np.where(BSP[n])[0][0])
        a, b = pair(n, ks, kk)
        rP.append(n); xs.append(a); xk.append(b); order.append('silent later' if ks > kk else 'silent earlier')
p1 = contrast('P1 silent vs speaking (same MCIS)', rP, xs, xk, 'silent', 'speaking')
RES17['P1'] = RES16.get('P1 silent vs speaking (same MCIS)')
order = np.array(order)
for o in ('silent later', 'silent earlier'):
    m = order == o
    contrast(f'P1 [{o}]', np.array(rP)[m], [x for x, t in zip(xs, m) if t], [x for x, t in zip(xk, m) if t],
             'silent', 'speaking')
    RES17[f'P1 {o}'] = RES16.get(f'P1 [{o}]')

# P2: one model over all context observations of L
orow, oX, otype = [], [], []
for n in range(N):
    for k in (0, 1):
        c, idx = SLOT[(n, 1, k)]
        if idx and VOK[n, k]:
            orow.append(n); oX.append(expr(c, idx)); otype.append('speaking' if BSP[n, k] else 'silent')
orow, otype = np.array(orow, int), np.array(otype)
if len(orow) >= 50:
    ig, _ = oof_ig(np.array(oX), orow)
    ig = ig / LN2
    for t in ('silent', 'speaking'):
        m = (otype == t) & np.isfinite(ig)
        lo, hi = ep_boot(lambda i: ig[m][i].mean(), src[orow][m])
        print(f"P2 all context observations, {t}: n={m.sum()} | IG {ig[m].mean():+.4f} bits [{lo:+.4f},{hi:+.4f}]")
        RES17[f'P2 all {t}'] = dict(n=int(m.sum()), IG=float(ig[m].mean()), lo=lo, hi=hi)
    pairs = [(np.where((orow == n) & (otype == 'silent'))[0], np.where((orow == n) & (otype == 'speaking'))[0]) for n in rP]
    d2 = np.array([ig[a[0]] - ig[b[0]] for a, b in pairs if len(a) and len(b)])
    ok = np.isfinite(d2)
    s2 = np.array([n for (a, b), n in zip(pairs, rP) if len(a) and len(b)])
    if ok.sum() >= 30:
        lo, hi = ep_boot(lambda i: d2[ok][i].mean(), src[s2][ok])
        print(f"P2 paired (P1 MCIS, one model): n={ok.sum()} | silent − speaking {d2[ok].mean():+.4f} bits [{lo:+.4f},{hi:+.4f}]")
        RES17['P2 paired'] = dict(n=int(ok.sum()), delta=float(d2[ok].mean()), lo=lo, hi=hi)

if p1 is None:
    dec = 'P1 UNINFORMATIVE (too few MCIS)'
elif p1[1] > 0:
    dec = 'constraint (iii) stays in the problem statement'
elif p1[0] > 0:
    dec = 'DIRECTIONAL -> (iii) becomes a reported finding with a caveat'
else:
    dec = 'constraint (iii) dropped'
print(f"\n== G17 decision (fixed rule): {dec} ==")
json.dump({k: ({a: (float(b) if isinstance(b, (np.floating, float)) else b) for a, b in v.items()} if v else v)
           for k, v in RES17.items()} | {'decision': dec}, open(f"{OUT_DIR}/g17_results.json", 'w'), indent=1, default=str)
print(f"saved {OUT_DIR}/g17_results.json")